In [1]:
model = "llama3.2:1B"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [2]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model="llama3.2:1b")

In [13]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)


In [14]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_collection(name="ai_model_book")

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(  client=client,

    collection_name="ai_model_book",

    embedding_function=embedding_model)

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = prompt | model | StrOutputParser()

In [16]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query, k=4)

print(docs)

[Document(id='300e1a43-5225-40b6-a9b6-b4a6e2c5f018', metadata={'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'page': 33, 'moddate': '2019-05-07T15:51:31+00:00', 'creationdate': '2019-05-07T15:51:31+00:00', 'total_pages': 510, 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'trapped': '/False', 'page_label': '8', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'source': './AI_Book.pdf'}, page_content='Types of Machine Learning Systems\nThere are so many different types of Machine Learning systems that it is useful to\nclassify them in broad categories based on:\n Whether or not they are trained with human supervision (supervised, unsuper\nvised, semisupervised, and Reinforcement Learning)\n Whether or not they can learn incrementally on the fly (online versus batch\nlearning)\n Whether they work by simply comparing new data points to known data points,\nor instead detect patterns in the training data and 

In [17]:
chain.invoke({"docs": format_docs(docs)})

'The main themes in these retrieved documents are:\n\n1. **Classification of Machine Learning Systems**: The documents discuss how Machine Learning systems can be categorized into various types based on factors such as:\n\t* Supervised vs. unsupervised learning\n\t* Training data type (supervised, unsupervised, semisupervised, and Reinforcement Learning)\n\t* Learning methods (batch vs. online, instance-based vs. model-based)\n2. **Machine Learning Fundamentals**: The documents provide an overview of the basics of Machine Learning, including:\n\t* The importance of data in Machine Learning\n\t* The difference between supervised and unsupervised learning\n\t* The role of algorithms in Machine Learning\n3. **Characteristics of Machine Learning Systems**: The documents highlight the key characteristics of different types of Machine Learning systems, including:\n\t* Model-based vs. instance-based learning\n\t* Batch vs. online learning\n\t* Supervised vs. unsupervised learning\n4. **Genera

In [18]:
# Simple stream the chain output
for chunk in chain.stream({"docs": format_docs(docs)}):
    print(chunk, end="", flush=True)

The main themes in the retrieved documents are:

1. **Classification of Machine Learning Systems**: The documents discuss how to categorize Machine Learning (ML) systems into broad categories based on their training methods, learning approaches, and generalization abilities.
2. **Types of Machine Learning Systems**: The documents highlight the four major categories of ML systems:
	* Supervised learning
	* Unsupervised learning
	* Semisupervised learning
	* Reinforcement learning
3. **Key Characteristics of ML Systems**: The documents emphasize the importance of considering the following factors when designing an ML system:
	* Training data and supervision
	* Learning approach (model-based or instance-based)
	* Generalization ability (instance-based or model-based)
4. **Performance Monitoring and Maintenance**: The documents discuss the need to monitor ML systems closely and promptly switch them off if performance drops, as well as react to abnormal data to prevent errors.
5. **Balancin

In [19]:
# More complex async event streaming
async for event in chain.astream_events(
    {"docs": format_docs(docs)}, version="v2"
):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

The main themes in these retrieved documents are:

1. **Classification of Machine Learning Systems**: The documents classify Machine Learning systems into four main categories based on training data, supervision, and learning mechanisms:
	* Supervised learning (with labels)
	* Unsupervised learning (without labels)
	* Semisupervised learning (with some supervision)
	* Reinforcement learning (no explicit supervision)
2. **Different Learning Mechanisms**: The documents highlight the differences between two primary learning mechanisms:
	* Instance-based learning: learns by comparing new data points to known data points
	* Model-based learning: learns a predictive model that generalizes to new instances
3. **Importance of Supervision and Monitoring**: The documents stress the importance of proper supervision and monitoring in Machine Learning systems, including:
	* Monitoring performance and switching off the system if necessary
	* Handling noisy or irrelevant data
	* Reacting to abnormal 

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [20]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever(search_kwargs={"k": 4})

# ADD HERE YOUR CODE
qa_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

In [21]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x1185d6170>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6'}}, model='llama3.2:1b')
| StrOutputParser()

In [22]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is a type of machine learning where the training data includes the desired solutions or labels, allowing the algorithm to learn without being labeled manually.'

In [23]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning system where the training data includes the desired solutions (labels) and the algorithm is trained to predict or classify the data based on these labels.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [ ]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

In [ ]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)